<a href="https://colab.research.google.com/github/sriharan17/SriflyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sriharan17/SriflyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row represents one pseudonymized content item (page).

The dataset contains 30,000 content items across 32 pseudonymized clients.

The starter CSV is a snapshot, not a daily panel. Its performance metrics are aggregated over a trailing 90-day window ending at export time.

For this contract, I use the observed 90-day snapshot and the embedded last-30-day versus previous-30-day comparison windows.

I will rank content items for refresh review based on observed decline signals. This is decision-support, not a prediction of Google's ranking algorithm.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
import pandas as pd

# Corrected URL: 'githubbusercontent' -> 'githubusercontent'
url = "https://raw.githubusercontent.com/sriharan17/SriflyrankAI/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:",df.shape)
print("Columns:")
for i ,col in enumerate(df.columns):
  print(i,col)

Shape: (30000, 44)
Columns:
0 content_id
1 client_id
2 search_volume
3 competition
4 competition_level
5 cpc
6 content_type
7 main_intent
8 word_count
9 char_count
10 provider_used
11 model_used
12 impressions_90d
13 clicks_90d
14 pageviews_90d
15 sessions_90d
16 users_90d
17 engaged_sessions_90d
18 ai_sessions_90d
19 scroll_events_90d
20 days_with_impressions
21 days_with_sessions
22 impressions_last_30d
23 clicks_last_30d
24 sessions_last_30d
25 impressions_prev_30d
26 clicks_prev_30d
27 sessions_prev_30d
28 content_age_days
29 age_tier
30 age_tier_order
31 days_since_last_update
32 freshness_tier
33 word_count_tier
34 char_count_tier
35 ctr
36 avg_position
37 engagement_rate
38 scroll_rate
39 ai_traffic_pct
40 impression_tier
41 position_tier
42 trend_direction
43 trend_pct


Features:
search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, content_age_days, days_since_last_update, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d.

Label:
trend_direction, interpreted as declining when trend_direction = "down".

Context:
content_id and client_id for identification and grouping; impressions_last_30d, clicks_last_30d, sessions_last_30d for describing the recent observation window.

Excluded:
trend_pct and trend_direction are excluded from model features because they directly encode the decline outcome. content_id and client_id are also excluded from model features because they are identifiers rather than meaningful predictive signals.

Five decision-time features:

1. search_volume — knowable at the decision moment because it describes the current search demand for the content topic.

2. competition — knowable at the decision moment because it describes the current competitive level for the topic.

3. cpc — knowable at the decision moment because the current keyword cost-per-click is available before deciding whether to refresh content.

4. content_age_days — knowable at the decision moment because the age of the content is already known when making the refresh decision.

5. days_since_last_update — knowable at the decision moment because the time since the previous content update is known before the decision.

In [12]:
# Deliberate leakage experiment
# trend_pct is derived from the outcome/trend and should NOT be used
# as a decision-time feature.

print("Label distribution:")
print(df["trend_direction"].value_counts())

print("\nLeakage column:")
print(df["trend_pct"].describe())

Label distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Leakage column:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "content_age_days",
    "days_since_last_update"
]

feature_frame = df[["content_id"] + features].copy()

print("Feature frame shape:", feature_frame.shape)
print("\nFeatures:")
print(feature_frame.head())

Feature frame shape: (30000, 6)

Features:
             content_id  search_volume  competition   cpc  content_age_days  \
0  content_304f48230142           10.0         0.67  2.05               187   
1  content_a1fb4e703a9e           90.0         0.01  0.05               445   
2  content_9aa793d4d895            0.0         0.00  0.00               141   
3  content_331d6c4de07b           10.0         0.00  0.00               463   
4  content_d99b7a2d90ca            0.0         0.00  0.00               263   

   days_since_last_update  
0                      20  
1                      25  
2                      20  
3                      22  
4                      14  


In [9]:
print("Rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\n90-day activity columns:")
print(df[[
    "impressions_90d",
    "clicks_90d",
    "sessions_90d"
]].describe())

Rows: 30000
Unique content items: 30000
Unique clients: 32

90-day activity columns:
       impressions_90d    clicks_90d  sessions_90d
count     30000.000000  30000.000000  30000.000000
mean       5200.366300     16.097333     37.066633
std       16838.019547     75.076958    107.069131
min           1.000000      0.000000      1.000000
25%          81.000000      0.000000      2.000000
50%         731.000000      1.000000      7.000000
75%        3615.250000      7.000000     27.000000
max      517715.000000   4178.000000   4345.000000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Query 1 — verify the grain
print("QUERY 1 — Grain")
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_id:", df["content_id"].duplicated().sum())


# Query 2 — verify the observation window
print("\nQUERY 2 — 90-day observation window")
print("Rows:", len(df))
print("impressions_90d available:", df["impressions_90d"].notna().sum())
print("clicks_90d available:", df["clicks_90d"].notna().sum())
print("sessions_90d available:", df["sessions_90d"].notna().sum())


# Query 3 — verify recent activity availability
print("\nQUERY 3 — Recent activity availability")
recent_available = (
    df["impressions_last_30d"].notna()
    & df["clicks_last_30d"].notna()
    & df["sessions_last_30d"].notna()
)

print("Rows with all three recent metrics available:", recent_available.sum())
print("Rows surviving filter:", len(df[recent_available]))

QUERY 1 — Grain
Rows: 30000
Unique content_id: 30000
Duplicate content_id: 0

QUERY 2 — 90-day observation window
Rows: 30000
impressions_90d available: 30000
clicks_90d available: 30000
sessions_90d available: 30000

QUERY 3 — Recent activity availability
Rows with all three recent metrics available: 30000
Rows surviving filter: 30000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset is a snapshot rather than a time-series panel, so it cannot establish long-term causal effects of refreshing content. The 90-day and 30-day windows overlap, so recent and previous-period metrics are not independent observations. The data also does not contain the actual content quality, Google ranking decisions, or the reason a page gained or lost traffic. Therefore, the features can support refresh prioritization but cannot prove that a refresh caused future performance changes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.